Gaussian elimination with partial pivoting

In [1]:
import numpy as np

def gaussian_elimination(A, b):
    n = len(b)
    Ab = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])

    for k in range(n):
        max_row = k + np.argmax(np.abs(Ab[k:, k]))
        Ab[[k, max_row]] = Ab[[max_row, k]]

        if abs(Ab[k, k]) < 1e-12:
            raise ValueError(f"Matrix is singular or nearly singular at pivot {k}")

        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - Ab[i, i+1:n] @ x[i+1:n]) / Ab[i, i]

    return x

In [ ]:
A = np.array([[2, 1, -1],
              [-3, -1, 2],
              [-2, 1, 2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

x = gaussian_elimination(A, b)
print("Solution:", x)
print("Residual (A@x - b):", A @ x - b)

Solution: [ 2.  3. -1.]
Residual (A@x - b): [0.0000000e+00 0.0000000e+00 8.8817842e-16]


In [12]:
A.shape

(3, 3)

In [19]:
np.hstack([A,b.reshape(-1,1)])

array([[  2.,   1.,  -1.,   8.],
       [ -3.,  -1.,   2., -11.],
       [ -2.,   1.,   2.,  -3.]])

In [9]:
b.shape

(3,)

In [18]:
b.reshape(-1,1).shape

(3, 1)

LU decomposition

In [20]:
def lu_decompose(A):
    n = A.shape[0]
    L = np.eye(n)
    U = A.astype(float).copy()
    P = np.eye(n)

    for k in range(n):
        max_row = k + np.argmax(np.abs(U[k:, k]))
        if max_row != k:
            U[[k, max_row]] = U[[max_row, k]]
            P[[k, max_row]] = P[[max_row, k]]
            if k > 0:
                L[[k, max_row], :k] = L[[max_row, k], :k]

        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]

    return P, L, U

def lu_solve(P, L, U, b):
    n = len(b)
    Pb = P @ b.astype(float)

    y = np.zeros(n)
    for i in range(n):
        y[i] = Pb[i] - L[i, :i] @ y[:i]

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]

    return x

In [21]:
A = np.array([[2, 1, -1],
              [-3, -1, 2],
              [-2, 1, 2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

P, L, U = lu_decompose(A)
x = lu_solve(P, L, U, b)

print("P =\n", P)
print("L =\n", np.round(L, 4))
print("U =\n", np.round(U, 4))
print("Solution:", x)
print("Residual (A@x - b):", A @ x - b)
print("Check P@A == L@U:", np.allclose(P @ A, L @ U))

P =
 [[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
L =
 [[ 1.      0.      0.    ]
 [ 0.6667  1.      0.    ]
 [-0.6667  0.2     1.    ]]
U =
 [[-3.     -1.      2.    ]
 [ 0.      1.6667  0.6667]
 [ 0.      0.      0.2   ]]
Solution: [ 2.  3. -1.]
Residual (A@x - b): [0.0000000e+00 0.0000000e+00 4.4408921e-16]
Check P@A == L@U: True


Cholesky decomposition

In [22]:
def cholesky(A):
    n = A.shape[0]
    L = np.zeros_like(A, dtype=float)

    for i in range(n):
        for j in range(i + 1):
            s = A[i, j] - L[i, :j] @ L[j, :j]
            if i == j:
                if s <= 0:
                    raise ValueError("Matrix is not positive definite")
                L[i, j] = np.sqrt(s)
            else:
                L[i, j] = s / L[j, j]

    return L

In [23]:
A = np.array([[4, 12, -16],
              [12, 37, -43],
              [-16, -43, 98]], dtype=float)
b = np.array([1, 2, 3], dtype=float)

L = cholesky(A)
print("L =\n", np.round(L, 4))
print("Check L @ L.T == A:", np.allclose(L @ L.T, A))

# Solve A x = b using forward + back substitution
y = np.linalg.solve(L, b)
x = np.linalg.solve(L.T, y)
print("Solution:", x)
print("Residual (A@x - b):", A @ x - b)

L =
 [[ 2.  0.  0.]
 [ 6.  1.  0.]
 [-8.  5.  3.]]
Check L @ L.T == A: True
Solution: [28.58333333 -7.66666667  1.33333333]
Residual (A@x - b): [ 3.55271368e-15 -2.50910404e-14 -1.28785871e-14]


 Least squares via normal equations

In [24]:
def least_squares_normal(A, b):
    AtA = A.T @ A
    Atb = A.T @ b
    return gaussian_elimination(AtA, Atb)

def ridge_regression(A, b, lam):
    n = A.shape[1]
    AtA = A.T @ A + lam * np.eye(n)
    Atb = A.T @ b
    L = cholesky(AtA)
    y = np.zeros(n)
    for i in range(n):
        y[i] = (Atb[i] - L[i, :i] @ y[:i]) / L[i, i]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - L.T[i, i+1:] @ x[i+1:]) / L.T[i, i]
    return x

In [25]:
np.random.seed(0)
N = 20
x = np.linspace(0, 5, N)
y = 2 * x + 1 + np.random.randn(N) * 0.5   # noisy line

# Design matrix: column of ones (intercept) + x (slope)
A = np.column_stack([np.ones(N), x])
b = y

# Ordinary least squares (normal equations)
w_ols = least_squares_normal(A, b)
print("OLS  (intercept, slope):", np.round(w_ols, 4))

# Ridge regression with small lambda
w_ridge = ridge_regression(A, b, lam=1.0)
print("Ridge(intercept, slope):", np.round(w_ridge, 4))

# Compare to NumPy's lstsq
w_np, *_ = np.linalg.lstsq(A, b, rcond=None)
print("NumPy (intercept, slope):", np.round(w_np, 4))

print("\nResidual OLS  :", np.round(A @ w_ols - b, 4)[:5], "...")
print("Check ridge finite:", np.all(np.isfinite(w_ridge)))

OLS  (intercept, slope): [1.569  1.8863]
Ridge(intercept, slope): [1.4112 1.9212]
NumPy (intercept, slope): [1.569  1.8863]

Residual OLS  : [-0.3131  0.339   0.0198 -0.6413 -0.4845] ...
Check ridge finite: True


Condition number

In [26]:
def condition_number(A):
    U, S, Vt = np.linalg.svd(A)
    return S[0] / S[-1]

In [27]:
A_good = np.array([[2.0, 1.0],
                   [1.0, 3.0]])

A_bad = np.array([[1.0, 1.0],
                  [1.0, 1.0001]])

print("A_good:\n", A_good)
print("Condition number:", np.round(condition_number(A_good), 4))
print("NumPy check:     ", np.round(np.linalg.cond(A_good), 4))

print("\nA_bad:\n", A_bad)
print("Condition number:", np.round(condition_number(A_bad), 4))
print("NumPy check:     ", np.round(np.linalg.cond(A_bad), 4))

A_good:
 [[2. 1.]
 [1. 3.]]
Condition number: 2.618
NumPy check:      2.618

A_bad:
 [[1.     1.    ]
 [1.     1.0001]]
Condition number: 40002.0001
NumPy check:      40002.0001


In [31]:
U,S,Vt=np.linalg.svd(A_good)

In [32]:
S

array([3.61803399, 1.38196601])

In [33]:
U


array([[-0.52573111, -0.85065081],
       [-0.85065081,  0.52573111]])

In [35]:
Vt

array([[-0.52573111, -0.85065081],
       [-0.85065081,  0.52573111]])